# E13：分布、频数和分组比较图

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：验证性与综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 从分析问题和变量类型出发选择合适的基础图形。
2. 使用 Matplotlib 对象式框架和 Seaborn 绘制并标注统计图。
3. 使用直方图、频数图和箱线图描述分布与组间差异。
4. 识别坐标轴截断、标签缺失和选择性展示带来的表达风险。
5. 对图形作出简洁、克制且可由当前样本支持的解释。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、图形和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的官方分析视图。
4. 提交前重启内核并按顺序运行全部单元，检查数据口径、标题、坐标轴和文字结论。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改作图代码后必须重新运行当前单元，图形才会更新。
- 图形异常时，先检查列名、数据类型、缺失值、类别顺序和当前使用的 `Axes`。
- 如果变量或图形状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 输入选择、核验与图形选择：20 分钟
- 数值分布与分箱比较：15 分钟
- 分类频数：15 分钟
- 分组分布比较：20 分钟
- 图形辨析：10 分钟
- 导出、说明与提交：10 分钟

## 主线输入：继续 E12 成果，官方分析视图兜底

课程准备代码优先读取 E12 个人成果，并把 `final_score` 规范为本章使用的 `python_score`；个人成果不存在时使用官方 `student_eda.csv`。个人成果一旦被选中但字段或口径核验失败，程序会停止，不会静默替换。路径选择、字段整理和核验代码由课程提供，本实验重点是可视化。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 有可靠的 E12 个人成果时保持 True；需要从官方分析视图开始时改为 False。
USE_STUDENT_E12_OUTPUT = True

chapter5_dir = Path(".").resolve()
notebook_names = ["e13.ipynb", "e13_inclass.ipynb"]
if not any((chapter5_dir / name).exists() for name in notebook_names):
    chapter5_dir = Path("lab/applied-stat-64/chapter5").resolve()

shared_dir = chapter5_dir.parents[1] / "shared"
student_e12_path = (
    chapter5_dir.parent
    / "chapter4"
    / "e12_output"
    / "e12_analysis_ready.csv"
)
official_path = shared_dir / "analysis" / "student_eda.csv"
output_dir = chapter5_dir / "e13_output"

if USE_STUDENT_E12_OUTPUT and student_e12_path.exists():
    source_path = student_e12_path
    raw_students = pd.read_csv(source_path)
    required_e12_columns = [
        "student_id",
        "gender",
        "major",
        "study_hours",
        "attendance_rate",
        "final_score",
        "data_literacy_score",
    ]
    missing_columns = [
        column
        for column in required_e12_columns
        if column not in raw_students.columns
    ]
    if missing_columns:
        raise ValueError(
            f"E12 个人成果缺少字段：{missing_columns}。"
            "请修正个人成果，或把 USE_STUDENT_E12_OUTPUT 改为 False。"
        )
    student_df = (
        raw_students[required_e12_columns]
        .rename(columns={"final_score": "python_score"})
        .copy()
    )
    print("输入来源：E12 个人成果")
else:
    if USE_STUDENT_E12_OUTPUT:
        print("未找到 E12 个人成果，改用官方分析视图。")
    source_path = official_path
    student_df = pd.read_csv(source_path)
    print("输入来源：官方分析视图")

expected_columns = [
    "student_id",
    "gender",
    "major",
    "study_hours",
    "attendance_rate",
    "python_score",
    "data_literacy_score",
]
major_order = [
    "applied_statistics",
    "statistical_computing",
    "data_science",
]
checks = {
    "字段及顺序正确": student_df.columns.tolist() == expected_columns,
    "记录数为 160": len(student_df) == 160,
    "student_id 唯一": student_df["student_id"].is_unique,
    "正式字段无缺失": not student_df.isna().any().any(),
    "性别取值正确": set(student_df["gender"]) == {"female", "male"},
    "专业取值正确": set(student_df["major"]) == set(major_order),
}
for check_name, passed in checks.items():
    print(check_name, "：", passed)
if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError("输入核验未通过：" + "、".join(failed))

student_df["major"] = pd.Categorical(
    student_df["major"],
    categories=major_order,
    ordered=True,
)
print("输入文件：", source_path)
print("数据形状：", student_df.shape)

## 任务 1：检查数据并匹配问题与图形

先运行代码确认变量类型、类别频数和数值摘要，再回答：

1. `python_score` 的分布适合用什么图？
2. 不同专业各有多少学生适合用什么图？
3. 不同专业的成绩分布是否相同适合用什么图？

不要先作图再猜问题；先确认观测单位、变量类型和缺失情况。

In [ ]:
print(student_df.dtypes)
print("\n专业频数：")
print(student_df["major"].value_counts().reindex(...))
print("\n数值摘要：")
print(
    student_df[
        [...]
    ].describe().round(2)
)

## 任务 2：绘制成绩分布并比较分箱

在同一行绘制两张 `python_score` 直方图：

- 左图使用 10 分宽的分箱；
- 右图使用 5 分宽的分箱。

两图必须使用相同的数据范围，并补全标题和坐标轴标签。比较细节变化，但不要把某一种分箱产生的小波动直接解释为稳定的分布结构。

In [ ]:
distribution_fig, axes = plt.subplots(
    1, 2, figsize=(10, 4), sharey=True
)

sns.histplot(
    data=student_df,
    x=...,
    bins=np.arange(40, 110, 10),
    color="#4C78A8",
    ax=axes[0],
)
axes[0].set(
    title=...,
    xlabel=...,
    ylabel=...,
)

sns.histplot(
    data=student_df,
    x=...,
    bins=np.arange(40, 105, 5),
    color="#59A14F",
    ax=axes[1],
)
axes[1].set(
    title=...,
    xlabel=...,
    ylabel=...,
)

distribution_fig.tight_layout()
plt.show()

## 任务 3：从频数表到频数图

先用 `value_counts()` 得到不同专业的精确人数，再用 `sns.countplot()` 绘制频数图。使用课程给出的 `major_order` 保持表格与图形顺序一致，并写清纵轴表示人数。

In [ ]:
major_counts = (
    student_df["major"].value_counts().reindex(...)
)
print(major_counts)

frequency_fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(
    data=student_df,
    x=...,
    order=...,
    color="#76B7B2",
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
ax.tick_params(axis="x", rotation=15)
frequency_fig.tight_layout()
plt.show()

## 任务 4：比较不同专业的成绩分布

在同一个 `Axes` 上叠加箱线图和个体点：

- 箱线图概括中位数、四分位范围，并关闭默认离群点标记；
- 个体点统一保留每名学生的位置，避免同一观测被重复绘制；
- 使用较浅的箱体和半透明个体点，避免相互遮挡。

图形差异是当前样本的描述，不等于已经完成总体差异检验。

In [ ]:
group_fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(
    data=student_df,
    x=...,
    y=...,
    order=major_order,
    color="#D9E6F2",
    width=0.55,
    showfliers=False,
    ax=ax,
)
sns.stripplot(
    data=student_df,
    x=...,
    y=...,
    order=major_order,
    color="#333333",
    alpha=0.35,
    size=3,
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
ax.tick_params(axis="x", rotation=15)
group_fig.tight_layout()
plt.show()

## 任务 5：修正一张容易误导的频数图

左图故意截断纵轴并省略标签，使 81 人与 79 人看起来差距很大。完成右图：

1. 纵轴从 0 开始；
2. 标明标题、横轴和纵轴；
3. 用一句话说明为什么不能根据左图声称两组人数差距很大。

In [ ]:
gender_order = ["female", "male"]
gender_counts = (
    student_df["gender"].value_counts().reindex(gender_order)
)
audit_fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 不规范示例：课程提供，只需观察。
axes[0].bar(
    gender_counts.index,
    gender_counts.values,
    color="#F28E8E",
)
axes[0].set_ylim(77, 83)
axes[0].set_title("Truncated axis")

# 请完成修正图。
axes[1].bar(
    gender_counts.index,
    gender_counts.values,
    color="#76B7B2",
)
axes[1].set(
    title=...,
    xlabel=...,
    ylabel=...,
)
axes[1].set_ylim(...)

audit_fig.tight_layout()
plt.show()

## 任务 6：导出代表性图形并完成核验

把任务 4 的分组分布图导出为 `e13_major_score_distribution.png`。课程检查只确认文件存在和基本数据口径；你还需要人工确认图片标题、坐标轴、类别顺序和个体点均清楚可读。

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
figure_path = output_dir / "e13_major_score_distribution.png"
group_fig.savefig(figure_path, dpi=150, bbox_inches="tight")

final_checks = {
    "记录数仍为 160": len(student_df) == 160,
    "专业顺序保持一致": (
        student_df["major"].cat.categories.tolist() == major_order
    ),
    "代表性图形已导出": figure_path.exists(),
}
for check_name, passed in final_checks.items():
    print(check_name, "：", passed)
if not all(final_checks.values()):
    raise ValueError("最终核验未通过。")
print("导出文件：", figure_path)

## 任务 7：图形说明与实验小结

用 Markdown 回答：

1. 两种分箱下，哪些总体特征相对稳定，哪些局部细节发生变化？
2. 三个专业的样本量和成绩分布有什么可直接描述的差异？
3. 为什么被箱线图规则识别为可能离群的观测不能直接当作错误数据删除？
4. 左侧截断纵轴的图形为什么容易误导？
5. 本实验是否使用了帮助文档、代码补全或 AI 工具？你如何核验其建议？

### 图形说明与实验小结

- ...
- ...
- ...
- ...
- ...